# Phase 2 - Live feed validation

Checks that `f360_raw.live_events` receives rows during an FPL poll window.
Run while Cloud Scheduler is firing `/webhook/fpl` for the current gameweek,
or POST to that endpoint manually from a shell.

In [ ]:
from google.cloud import bigquery
from etl.config import settings

client = bigquery.Client(project=settings.gcp_project_id, location=settings.gcp_bq_location)

def count_live():
    sql = f"""
        SELECT COUNT(*) AS n
        FROM `{settings.gcp_project_id}.{settings.bq_dataset_raw}.live_events`
    """
    return list(client.query(sql).result())[0].n

print(f"live_events rows: {count_live()}")

In [ ]:
import time

samples = []
for i in range(10):
    n = count_live()
    samples.append(n)
    print(f"t={i*30}s live_events={n}")
    time.sleep(30)

assert samples[-1] > 0, "expected live_events to grow during FPL polling"
print("OK")